In [1]:
from chatterbox.tts import ChatterboxTTS
from huggingface_hub import hf_hub_download
import torch
import torchaudio
import os
import gc
from IPython.display import Audio
from safetensors.torch import save_file, load_file


from warnings import filterwarnings

filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

c:\Users\LENOVO\anaconda3\envs\kcvtts\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
original_load = torch.load

def safe_cpu_load(*args, **kwargs):
    kwargs['map_location'] = torch.device('cpu')
    return original_load(*args, **kwargs)

torch.load = safe_cpu_load

In [ ]:
MODEL_REPO = "grandhigh/Chatterbox-TTS-Indonesian"
CHECKPOINT_FILENAME = "t3_cfg.safetensors"

model = ChatterboxTTS.from_pretrained(device="cpu")

checkpoint_path = hf_hub_download(repo_id=MODEL_REPO, filename=CHECKPOINT_FILENAME)
t3_state = load_file(checkpoint_path, device="cpu")

model.t3.load_state_dict(t3_state)


In [ ]:
output_dir = "chatterbox_indonesia"
os.makedirs(output_dir, exist_ok=True)

full_state_dict = model.t3.state_dict()

save_path = os.path.join(output_dir, "model_tts.safetensors")

save_file(full_state_dict, save_path)


In [3]:
model = ChatterboxTTS.from_pretrained(device="cpu") 

model_path = "chatterbox_indonesia/model_tts.safetensors"
t3_state = load_file(model_path, device="cpu")

model.t3.load_state_dict(t3_state)

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /ResembleAI/chatterbox/resolve/main/ve.pt (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: edcf3173-8535-4fdb-a26b-44533bdf8287)')' thrown while requesting HEAD https://huggingface.co/ResembleAI/chatterbox/resolve/main/ve.pt
Retrying in 1s [Retry 1/5].


loaded PerthNet (Implicit) at step 250,000


<All keys matched successfully>

In [6]:
file_ref = '../data/suara_hartmann.wav'
# text = 'sebuah laptop hitam dengan layar yang menampilkan beberapa teks dan keyboard hitam dengan huruf putih. pada layarnya terdapat teks yang menyatakan bahwa disk pengisian daya siap, dapat digunakan, dan berfungsi dengan baik. ada juga teks lain yang menyebutkan bahwa pengaturan telah dikonfigurasi dan perlu diulang untuk melanjutkan. sebagian dari keyboard yang terlihat termasuk tombol 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, q, w, e, r, t, y, u, i, o, p, z, x, c, v, f, g, h, j, k, l, m, n, b, v, tab, dan spasi.'
# text = "seorang pria sedang duduk dan bekerja di depan komputer laptop di sebuah ruangan dengan banyak kursi dan meja kosong lainnya. Pria itu mengenakan celana panjang hitam dan sepatu putih. Kemejanya berwarna krem ​​dan ia memiliki rambut pendek. Sebuah tas laptop tergeletak di lantai di bawah meja di dekat kakinya."
text = "aku sayang andra"
with torch.inference_mode():
    wav_audio_clone = model.generate(
        text,
        audio_prompt_path=file_ref
    )

display(Audio(wav_audio_clone.numpy(), rate=model.sr))


Sampling:   5%|▍         | 49/1000 [00:06<02:14,  7.05it/s]


In [7]:
audio_tensor = wav_audio_clone.unsqueeze(0) if wav_audio_clone.dim() == 1 else wav_audio_clone

nama_file = "caption_hartmann_andr.mp3"
torchaudio.save(nama_file, audio_tensor, model.sr)